# 01 · Exploración del subconjunto de Amazon Reviews 2023

Categoría *Cell Phones & Accessories*: los 40 productos más calificados de 8 tipos, con hasta 200 reseñas por producto elegidas por muestreo de reservorio (uniforme en el tiempo). Se descarga con `python descargar_datos.py` (lee 13 GB en streaming y guarda ~7 MB).

In [1]:
import sys, json
from pathlib import Path
RAIZ = Path.cwd().parent
sys.path.insert(0, str(RAIZ / "src"))
import pandas as pd
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 160)
from resenas import config
from resenas.preparar import cargar_resenas
r = cargar_resenas()
print(f'{len(r):,} reseñas · {r.parent_asin.nunique()} productos · {r.marca.nunique()} marcas')
r.head(3)[['id','estrellas','tipo_producto','marca','titulo_resena','texto','fecha']]

55,082 reseñas · 320 productos · 163 marcas


,id,estrellas,tipo_producto,marca,titulo_resena,texto,fecha
0,R000000,5,Auriculares Bluetooth,Motorola,Motorola H720 89382N Bluetooth Headset,Great headset. I have owned other Motorola headsets and the quality remains the same.,2010-07-25 14:26:40
1,R000001,2,Auriculares Bluetooth,Motorola,flawed,I prefer the boom-type blue tooth ear pieces so tried this one. Stopped recharging after 2-3 weeks. Am now using the...,2010-08-03 19:27:08
2,R000002,5,Auriculares Bluetooth,Motorola,Very please with this product,"After reviewing option, I chose to purchase an earlier version for a lower price. The product is good and the 720 ri...",2010-09-06 04:26:54


## Tipos de producto y estrellas
Las reseñas de 5★ son ~2/3: por eso la muestra etiquetada se estratificó por estrellas (si no, habría muy pocas quejas que evaluar).

In [2]:
t = r.groupby('tipo_producto').agg(resenas=('id','size'), productos=('parent_asin','nunique'),
    marcas=('marca','nunique'), estrellas_medias=('estrellas','mean'), pct_1_2=('estrellas', lambda s: (s<=2).mean()*100))
t.round(2).sort_values('estrellas_medias')

,resenas,productos,marcas,estrellas_medias,pct_1_2
tipo_producto,,,,,
Teléfonos,4105,40,11,3.91,22.31
Auriculares Bluetooth,6696,40,27,4.03,18.58
Accesorios para auto,7651,40,27,4.07,18.99
Protectores de pantalla,7271,40,16,4.09,19.34
Cargadores y power banks,7875,40,24,4.14,18.53
Accesorios de smartwatch,7258,40,29,4.23,15.03
Cables y adaptadores,7081,40,25,4.25,16.01
Fundas,7145,40,19,4.40,11.03


In [3]:
pd.crosstab(r.tipo_producto, r.estrellas, normalize='index').mul(100).round(1)

estrellas,1,2,3,4,5
tipo_producto,,,,,
Accesorios de smartwatch,10.2,4.9,5.8,9.7,69.5
Accesorios para auto,13.6,5.4,6.5,9.6,64.9
Auriculares Bluetooth,12.1,6.5,7.5,14.0,60.0
Cables y adaptadores,12.7,3.3,3.5,7.7,72.8
Cargadores y power banks,13.7,4.8,4.6,8.0,68.9
Fundas,7.3,3.7,4.7,10.1,74.1
Protectores de pantalla,13.0,6.3,5.7,8.2,66.8
Teléfonos,16.3,6.0,6.7,12.4,58.6


## Largo de las reseñas y años

In [4]:
print(r.palabras.describe().round(1))
print('Entre 5 y 150 palabras:', round(r.palabras.between(5,150).mean()*100,1), '%')
r.fecha.dt.year.value_counts().sort_index().to_frame('resenas').T

count    55082.0
mean        32.8
std         42.4
min          1.0
25%         10.0
50%         21.0
75%         41.0
max       1770.0
Name: palabras, dtype: float64
Entre 5 y 150 palabras: 90.1 %


fecha,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
resenas,5,13,36,156,286,654,1329,2825,3985,8535,12918,12859,8588,2893


## Reseñas en español
Amazon EE. UU. tiene clientes hispanohablantes: una heurística de palabras frecuentes (`texto.es_espanol`) encuentra ~2,5 % de reseñas en español, sobre todo de teléfonos comprados para Latinoamérica. Con ellas se armó el conjunto de evaluación en español (80 reseñas).

In [5]:
from resenas.texto import es_espanol, texto_completo
r['es'] = [es_espanol(texto_completo(a, b)) for a, b in zip(r.titulo_resena, r.texto)]
print(r.es.sum(), 'reseñas en español')
r.groupby('tipo_producto').es.mean().mul(100).round(1).sort_values(ascending=False).to_frame('% en español')

1361 reseñas en español


,% en español
tipo_producto,
Teléfonos,13.3
Auriculares Bluetooth,2.0
Protectores de pantalla,1.8
Accesorios de smartwatch,1.7
Accesorios para auto,1.5
Cables y adaptadores,1.4
Fundas,1.4
Cargadores y power banks,1.4


In [6]:
# El texto de las reseñas no se muestra ni se versiona (licencia de los datos): solo metadatos.
r[r.es].sample(6, random_state=3)[['estrellas','tipo_producto','palabras']]

,estrellas,tipo_producto,palabras
45098,5,Protectores de pantalla,4
27931,5,Fundas,9
13306,5,Accesorios de smartwatch,6
22370,5,Teléfonos,8
30444,5,Teléfonos,11
31024,4,Teléfonos,21


## Muestra etiquetada a mano
620 reseñas (dev 300 / test 320) + 80 en español. Guía: `docs/GUIA_ANOTACION.md`.

In [7]:
from resenas.etiquetas import leer_jsonl
import collections
for nombre, ruta in [('dev', config.ETIQUETAS_DEV), ('test', config.ETIQUETAS_TEST), ('español', config.ETIQUETAS_ES)]:
    e = leer_jsonl(ruta)
    c = collections.Counter(m['aspecto'] for d in e for m in d['aspectos'])
    print(f"{nombre:8} {len(e)} reseñas · {sum(c.values())} menciones · {sum(not d['aspectos'] for d in e)} sin aspectos · ", dict(c.most_common(5)))

dev      300 reseñas · 491 menciones · 15 sin aspectos ·  {'calidad': 97, 'ajuste': 82, 'diseno': 58, 'facilidad_uso': 55, 'precio': 49}
test     320 reseñas · 546 menciones · 11 sin aspectos ·  {'ajuste': 102, 'calidad': 99, 'facilidad_uso': 63, 'diseno': 49, 'carga': 46}
español  80 reseñas · 110 menciones · 14 sin aspectos ·  {'calidad': 23, 'ajuste': 13, 'facilidad_uso': 12, 'diseno': 12, 'precio': 10}
